# 27 · Semantic Caching

**Idea:** an exact-match cache misses *"what is RAG?"* vs *"explain RAG"* — same intent, different string. A semantic cache keys on *meaning*: embed each query, and on a new query return a cached answer when cosine similarity to a stored query exceeds a threshold. This cuts cost and latency for the long tail of near-duplicate questions any real RAG service receives.

> Embeds queries with Ollama. No cloud keys required.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Store one answer, then query with a **paraphrase**. An exact cache would miss; the semantic cache returns a hit because the embeddings are close.

In [ ]:
from rag_lab import SemanticCache

cache = SemanticCache(threshold=0.85, provider='ollama')
cache.put('What is retrieval-augmented generation?', 'RAG retrieves context and feeds it to an LLM to ground its answer.')

print('paraphrase ->', cache.get('Can you explain RAG?'))
print('unrelated  ->', cache.get('What is the capital of France?'))
print('stats:', cache.stats())

Wire it in front of the real pipeline: **check the cache first**, only run `rag_answer` (retrieve + generate) on a miss, then store the fresh answer. The second, paraphrased call skips generation entirely.

In [ ]:
from rag_lab import load_markdown, NumpyStore, rag_answer
from study_buddy import CostTracker

store = NumpyStore(provider='ollama'); store.add(load_markdown())
tracker = CostTracker()

def answer(query):
    cached = cache.get(query)
    if cached is not None:
        return cached, 'CACHE HIT'
    res = rag_answer(query, store, provider='ollama', k=4, tracker=tracker)
    cache.put(query, res.answer)
    return res.answer, 'GENERATED'

a1, s1 = answer('What is hybrid search?')
print(s1, '->', a1[:100], '...')
a2, s2 = answer('Explain hybrid search to me')
print(s2, '->', a2[:100], '...')
print('\ncache:', cache.stats(), '| generation cost:', tracker.summary())

The `threshold` is the key safety dial: too low returns stale/wrong answers for merely-similar questions; too high degrades to an exact-match cache. Inspect its effect.

In [ ]:
for t in (0.75, 0.85, 0.95):
    c = SemanticCache(threshold=t, provider='ollama')
    c.put('What is hybrid search?', 'combines BM25 and vectors')
    hit = c.get('what is keyword search?')
    print(f'threshold={t} -> off-topic query returns:', 'HIT (risky)' if hit else 'miss (safe)')

**Takeaway:** a semantic cache is a cheap multiplier on a RAG service — it collapses the long tail of reworded questions into one paid generation. Tune `threshold` conservatively; a wrong cache hit is worse than a cache miss.

Next up → `28_long_context_vs_rag.ipynb`, the "why retrieve at all?" question.